# Common OOP Patterns

AI generates these patterns frequently. Recognizing them helps you understand code faster.

## Pattern 1: Factory Methods

Alternative ways to create objects, often named `from_*`.

In [ ]:
from datetime import datetime

class User:
    def __init__(self, name: str, email: str, created_at: datetime):
        self.name = name
        self.email = email
        self.created_at = created_at
    
    @classmethod
    def from_dict(cls, data: dict) -> "User":
        """Create User from dictionary."""
        return cls(
            name=data["name"],
            email=data["email"],
            created_at=datetime.fromisoformat(data["created_at"])
        )
    
    @classmethod
    def create_guest(cls) -> "User":
        """Create a guest user."""
        return cls("Guest", "guest@example.com", datetime.now())

# Different ways to create users
user1 = User("Alice", "alice@example.com", datetime.now())
user2 = User.from_dict({"name": "Bob", "email": "bob@example.com", "created_at": "2024-01-01T00:00:00"})
user3 = User.create_guest()

print(f"Regular: {user1.name}")
print(f"From dict: {user2.name}")
print(f"Guest: {user3.name}")

## Pattern 2: Builder Pattern

Chain method calls to build an object step by step.

In [ ]:
class QueryBuilder:
    """Build SQL queries with method chaining."""
    
    def __init__(self):
        self._select = "*"
        self._from = None
        self._where = []
        self._order_by = None
        self._limit = None
    
    def select(self, *columns) -> "QueryBuilder":
        self._select = ", ".join(columns)
        return self  # Return self for chaining!
    
    def from_table(self, table: str) -> "QueryBuilder":
        self._from = table
        return self
    
    def where(self, condition: str) -> "QueryBuilder":
        self._where.append(condition)
        return self
    
    def order_by(self, column: str) -> "QueryBuilder":
        self._order_by = column
        return self
    
    def limit(self, n: int) -> "QueryBuilder":
        self._limit = n
        return self
    
    def build(self) -> str:
        parts = [f"SELECT {self._select}", f"FROM {self._from}"]
        if self._where:
            parts.append(f"WHERE {' AND '.join(self._where)}")
        if self._order_by:
            parts.append(f"ORDER BY {self._order_by}")
        if self._limit:
            parts.append(f"LIMIT {self._limit}")
        return " ".join(parts)

# Method chaining in action
query = (
    QueryBuilder()
    .select("name", "email")
    .from_table("users")
    .where("active = true")
    .where("age > 18")
    .order_by("name")
    .limit(10)
    .build()
)

print(query)

## Pattern 3: Singleton

Ensure only one instance exists. Often used for config, logging, database connections.

In [ ]:
# Simple singleton using module-level instance
class DatabaseConnection:
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance
    
    def __init__(self):
        if self._initialized:
            return
        self._initialized = True
        self.connection_string = "postgresql://localhost/db"
        print("Database connection created!")

# Both variables point to the SAME instance
db1 = DatabaseConnection()
db2 = DatabaseConnection()

print(f"Same instance? {db1 is db2}")  # True
print(f"Connection: {db1.connection_string}")

## Pattern 4: Strategy Pattern

Pass different behaviors as objects (or functions).

In [ ]:
from typing import Protocol, List

# Strategy interface (using Protocol for duck typing)
class SortStrategy(Protocol):
    def sort(self, data: List[int]) -> List[int]: ...

class AscendingSort:
    def sort(self, data: List[int]) -> List[int]:
        return sorted(data)

class DescendingSort:
    def sort(self, data: List[int]) -> List[int]:
        return sorted(data, reverse=True)

class Sorter:
    def __init__(self, strategy: SortStrategy):
        self.strategy = strategy
    
    def sort(self, data: List[int]) -> List[int]:
        return self.strategy.sort(data)

# Use different strategies
data = [3, 1, 4, 1, 5, 9, 2, 6]

asc_sorter = Sorter(AscendingSort())
desc_sorter = Sorter(DescendingSort())

print(f"Ascending: {asc_sorter.sort(data)}")
print(f"Descending: {desc_sorter.sort(data)}")

## Pattern 5: Repository Pattern

Abstract data storage operations. Very common in AI-generated web apps.

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional, List

@dataclass
class User:
    id: int
    name: str
    email: str

class UserRepository(ABC):
    """Abstract repository - defines the interface."""
    
    @abstractmethod
    def get(self, id: int) -> Optional[User]: ...
    
    @abstractmethod
    def save(self, user: User) -> None: ...
    
    @abstractmethod
    def all(self) -> List[User]: ...

class InMemoryUserRepository(UserRepository):
    """Concrete implementation - stores in memory."""
    
    def __init__(self):
        self._users = {}
    
    def get(self, id: int) -> Optional[User]:
        return self._users.get(id)
    
    def save(self, user: User) -> None:
        self._users[user.id] = user
    
    def all(self) -> List[User]:
        return list(self._users.values())

# Usage - can swap implementations easily
repo = InMemoryUserRepository()
repo.save(User(1, "Alice", "alice@example.com"))
repo.save(User(2, "Bob", "bob@example.com"))

print(f"User 1: {repo.get(1)}")
print(f"All users: {repo.all()}")

## Pattern 6: Dependency Injection

Pass dependencies in rather than creating them inside the class.

In [ ]:
# Bad: Hard-coded dependency
class BadUserService:
    def __init__(self):
        self.repo = InMemoryUserRepository()  # Can't change this!

# Good: Dependency injection
class GoodUserService:
    def __init__(self, repo: UserRepository):  # Pass dependency in
        self.repo = repo
    
    def get_user(self, id: int) -> Optional[User]:
        return self.repo.get(id)
    
    def create_user(self, name: str, email: str) -> User:
        user_id = len(self.repo.all()) + 1
        user = User(user_id, name, email)
        self.repo.save(user)
        return user

# Can use different repositories
repo = InMemoryUserRepository()
service = GoodUserService(repo)

user = service.create_user("Charlie", "charlie@example.com")
print(f"Created: {user}")
print(f"Retrieved: {service.get_user(user.id)}")

## Summary: Recognizing Patterns in AI Code

| Pattern | When You See | Purpose |
|---------|-------------|--------|
| Factory | `@classmethod` named `from_*` | Alternative constructors |
| Builder | Methods returning `self` | Fluent configuration |
| Singleton | `_instance` class variable | Single shared instance |
| Strategy | Interface + multiple implementations | Swappable algorithms |
| Repository | Abstract CRUD operations | Data access abstraction |
| DI | Dependencies passed to `__init__` | Loose coupling, testability |

## Module Complete!

You now understand:
- How to read class definitions
- Inheritance and `super()`
- Dataclasses for less boilerplate
- Common OOP patterns

Next module: Functions!